# Pasquereau & Turner (2013) on DANDI:001636

Paper: Pasquereau B, Turner RS (2013). Primary motor cortex of the parkinsonian monkey: altered
neuronal responses to muscle stretch. *Front. Syst. Neurosci.* 7:98,
[doi:10.3389/fnsys.2013.00098](https://doi.org/10.3389/fnsys.2013.00098).

Dataset: [DANDI:001636](https://dandiarchive.org/dandiset/001636/0.261124.1906), version
`0.261124.1906` (doi:10.48324/dandi.001636/0.261124.1906). This version was asked for in the
prompt of this attempt; it is also the newest published version (published 2026-11-24). The
paper cites no version.

Claims: `claims/pasquereau-2013.yaml`, written from the full text in PubMed Central
(PMC3840326), its Methods, Results, Table 1, figure legends and supplementary data sheet
(DataSheet1.PDF), whose table gives the results for each monkey.

The files of monkey L mark no torque perturbation, so every claim is tested on monkey V
(Venus, elbow), as the claims file says.

| Claim | Tested or blocked |
|---|---|
| cell-counts | tested |
| monkey-l-perturbations | blocked, data missing |
| baseline-rate | tested |
| ptn-faster-than-csn | tested |
| antidromic-latency | tested |
| false-positive-rate | computed; blocked, methods underspecified (the false-positive rate is far from the paper's) |
| torque-related-fractions, prevalence-unchanged, ptn-more-responsive, one-direction-half, excitatory-majority, magnitude-unchanged, venus-response-magnitudes, latency-shorter, latency-not-training, fwhm-shorter-ptn, dsi-reduced, venus-dsi, dsi-not-baseline, velocity-matched-magnitude, session-correlations | blocked through false-positive-rate; their results are computed and shown, not deciding |
| kinematics-slower | tested |
| kinematic-latency-unchanged | tested |
| emg-m2-larger | tested |
| emg-duration-longer | tested |
| emg-onset-unchanged | tested |
| emg-m1-weak | tested |
| trial-correlations | blocked, methods underspecified |

Revised after the first round of verification: the source files within each file, the angle's
scale, the Leu torque command, the blocked claims under the `trial_sd` reading, variants of the
kinematic onset and the EMG claims at matched torque commands were added.

Method details come from the claims file, which cites the Methods; details it lacks are taken
from the paper and cited in `pasq.py`'s docstring and below.

In [ ]:
import time
import warnings

import numpy as np
import pandas as pd
import pynwb
import remfile
import h5py
import matplotlib.pyplot as plt
import statsmodels.api as sm
import statsmodels.formula.api as smf
from scipy import stats

import pasq

warnings.filterwarnings("ignore", category=RuntimeWarning)
pd.set_option("display.width", 160)
pd.set_option("display.max_columns", 30)
PRE_COLOR, POST_COLOR = "#2a78d6", "#eb6834"
t0 = time.time()
venus_urls = pasq.asset_urls("sub-Venus")
leu_urls = pasq.asset_urls("sub-Leu")
print(f"{len(venus_urls)} Venus files, {len(leu_urls)} Leu files at version {pasq.DANDISET_VERSION}")

## What one file holds

One Venus file opened with pynwb: its acquisition and processing series, its interval tables
with every column, and the units table's columns.

In [ ]:
path, url = venus_urls[0]
with pynwb.NWBHDF5IO(file=h5py.File(remfile.File(url), "r"), load_namespaces=True) as io:
    nwb = io.read()
    print(path)
    for name, ts in nwb.acquisition.items():
        print(f"acquisition {name}: {ts.data.shape}, unit {ts.unit}")
    for mod_name, mod in nwb.processing.items():
        for name, obj in mod.data_interfaces.items():
            print(f"processing/{mod_name}/{name}: {type(obj).__name__}")
    for name, table in nwb.intervals.items():
        print(f"intervals {name}: {len(table)} rows, columns {list(table.colnames)}")
    print("units columns", list(nwb.units.colnames))
    print(nwb.units.to_dataframe().drop(columns=["spike_times", "obs_intervals", "electrodes"]).T)

## Loading

Every Venus file is streamed once; only windows around each perturbation are kept (spikes from
-1500 to +500 ms, angle -200 to +400 ms, EMG -300 to +300 ms, torque command -50 to +100 ms, and the file's ElbowVelocity series over the angle's window).
The angle at the trials table's voluntary movement onset and end is kept to check the angle's scale.
The Leu files are read for their trials' perturbation columns, torque command and units table.

In [ ]:
venus = pasq.load_all(venus_urls)
leu = pasq.load_meta_all(leu_urls)
print(f"loaded in {time.time() - t0:.0f} s")

In [ ]:
cells = pasq.cell_responses(venus)
cells["unit"] = cells.groupby("file").cumcount()
cells["studied"] = (cells.n_fle >= pasq.MIN_PERTURBATIONS) & (cells.n_ext >= pasq.MIN_PERTURBATIONS)
studied = cells[cells.studied].copy()
studied["state"] = np.where(studied.post, "post", "pre")
PTN, CSN = "pyramidal_tract_neuron", "corticostriatal_neuron"
groups = {"M1": studied.index == studied.index, "PTN": (studied.ptype == PTN).values,
          "CSN": (studied.ptype == CSN).values}
first_day = pd.to_datetime(cells.date).min()
kin_trials, kin_sessions = pasq.session_kinematics(venus)

## Data check

In [ ]:
all_units = pd.concat([x["units"].assign(subject="Venus", path=x["path"]) for x in venus]
                      + [x["units"].assign(subject="Leu", path=x["path"]) for x in leu], ignore_index=True)
print("units by subject, MPTP state and projection type")
print(all_units.groupby(["subject", "post", "ptype"]).size().to_string())
print("unit names repeated across files:", all_units.unit_name.duplicated().sum())
fname_post = all_units.path.str.contains("PostMPTP")
print("units whose is_post_mptp disagrees with the file name:", int((fname_post != all_units.post).sum()))

pert = pd.concat([x["pert"].assign(file=i, post=bool(x["units"].post.iloc[0])) for i, x in enumerate(venus)])
print("\nVenus perturbations by state and direction")
print(pert.groupby(["post", "torque"]).size().to_string())
print("Venus files without any perturbation:", sum(len(x["pert"]) == 0 for x in venus))
print("Leu trials by torque_perturbation_type:",
      pd.Series(np.concatenate([x["torque_type"] for x in leu])).value_counts().to_dict())
print("Leu perturbation onset times that are not NaN:", int(sum(np.isfinite(x["onset"]).sum() for x in leu)))

dates = pd.to_datetime(cells.date)
print("\nVenus recording dates: pre", dates[~cells.post].min().date(), "to", dates[~cells.post].max().date(),
      "; post", dates[cells.post].min().date(), "to", dates[cells.post].max().date())
delay = (pert.onset - pert.center)
print("perturbation onset after centre target appearance (s): median %.3f, 5th to 95th percentile %.3f to %.3f"
      % (delay.median(), delay.quantile(0.05), delay.quantile(0.95)))
print("units with fewer observed perturbations than their file has:",
      int((cells.n_fle + cells.n_ext < [len(venus[f]["pert"]) for f in cells.file]).sum()))

ang = np.concatenate([x["angle"].ravel() for x in venus if x["angle"] is not None and len(x["pert"])])
print("\nelbow angle around perturbations (degrees): %.1f to %.1f" % (np.nanmin(ang), np.nanmax(ang)))
emg_vals = {ch: np.concatenate([x["emg"][ch].ravel() for x in venus if ch in x["emg"]]) for ch in pasq.EMG_CHANNELS}
for ch, v in emg_vals.items():
    print(f"{ch}: {sum(ch in x['emg'] for x in venus)} files, values {np.nanmin(v):.0f} to {np.nanmax(v):.0f} (a.u.)")
print("SDF kernel: gamma k = %.1f, theta = %.0f ms, mode %.0f ms, mean %.0f ms; t threshold %.3f"
      % (pasq.GAMMA_K, pasq.GAMMA_THETA_MS, (pasq.GAMMA_K - 1) * pasq.GAMMA_THETA_MS,
         pasq.GAMMA_K * pasq.GAMMA_THETA_MS, pasq.T_CRIT))

### Recordings deposited twice

Each trial's spikes, relative to its trial start, and each perturbation's angle and EMG windows
are compared across all Venus files.

In [ ]:
from collections import defaultdict
from itertools import combinations

def shared(keys_by_file):
    owners = defaultdict(set)
    for f, keys in keys_by_file.items():
        for k in keys:
            owners[k].add(f)
    pairs = defaultdict(int)
    for fs in owners.values():
        for a, b in combinations(sorted(fs), 2):
            pairs[(a, b)] += 1
    return pairs

spike_keys = {}
for fi, x in enumerate(venus):
    keys = []
    for ui, st in enumerate(x["spikes_all"]):
        for a, b in zip(x["trials"].start, x["trials"].stop):
            s = np.round(st[(st >= a) & (st < b)] - a, 4)
            if len(s) >= 5:
                keys.append(tuple(s))
    spike_keys[fi] = keys
analog_keys = {}
for fi, x in enumerate(venus):
    keys = []
    for name, arr in [("angle", x["angle"])] + list(x["emg"].items()):
        if arr is None:
            continue
        keys += [(name,) + tuple(np.round(w[::7], 2)) for w in arr if not np.isnan(w).any()]
    analog_keys[fi] = keys
print("pairs of units sharing a trial's spike train:", len(shared(spike_keys)))
print("pairs of files sharing an angle or EMG window:", len(shared(analog_keys)))

### Files made of several source files

The trials table's `source_file` names the lab file each trial came from. Files holding more
than one source file are listed, and their source files' trials compared with each other.

In [ ]:
src_files = pd.concat([x["trials"][["source_file"]].assign(file=i) for i, x in enumerate(venus)])
per_file = src_files.groupby("file").source_file.unique()
multi = per_file[per_file.map(len) > 1]
print("Venus files holding trials of more than one source file:", len(multi), "of", len(venus))
for f, s in multi.items():
    print(f"  {venus[f]['path'].split('/')[-1][:62]}: {list(s)}; units {list(venus[f]['units'].unit_name)}")
print("source_file values found in more than one file:", int((src_files.groupby("source_file").file.nunique() > 1).sum()))
repeated = 0
for f in multi.index:
    x, t = venus[f], venus[f]["trials"]
    for st in x["spikes_all"]:
        seen = {}
        for a, b, sf in zip(t.start, t.stop, t.source_file):
            k = tuple(np.round(st[(st >= a) & (st < b)] - a, 4))
            if len(k) >= 5 and seen.setdefault(k, sf) != sf:
                repeated += 1
    seen = {}
    for w, sf in zip(x["angle"] if x["angle"] is not None else [], x["pert"].source_file):
        if not np.isnan(w).any():
            k = tuple(np.round(w[::7] - w[0], 2))
            if seen.setdefault(k, sf) != sf:
                repeated += 1
print("trials repeated between the source files of one file (spike trains or angle windows):", repeated)

### Clocks

The torque command, the elbow angle and the spikes are checked against the trials table's
perturbation onsets. The command should leave rest a few ms after onset and the joint should move
in the torque's direction after it. For the spikes, a contrast: threshold crossings by the claims
file's detection after the real onset against a pseudo onset 600 ms earlier and against trials
without a perturbation aligned where the file's perturbations fall.

In [ ]:
zt = -pasq.TORQUE_WINDOW_MS[0]
for post in [False, True]:
    cmd = []
    for x in venus:
        if x["torque_cmd"] is None or bool(x["units"].post.iloc[0]) != post or len(x["pert"]) == 0:
            continue
        s = np.where(x["pert"].torque.values == "flexion", 1.0, -1.0)[:, None]
        cmd.append(s * (x["torque_cmd"] - x["torque_cmd"][:, :zt].mean(1, keepdims=True)))
    m = np.nanmean(np.vstack(cmd), 0)
    half = np.where(m[zt:] > m[zt:zt + 50].max() / 2)[0][0]
    leave = np.where(m[zt:] > 0.05 * m[zt:zt + 50].max())[0][0]
    print(f"{'post' if post else 'pre'}-MPTP: mean command leaves rest (5% of pulse) at {leave} ms and reaches half its peak at {half} ms after onset")
za = -pasq.ANGLE_WINDOW_MS[0]
for dname, sgn in [("flexion", 1), ("extension", -1)]:
    a = np.vstack([x["angle"][x["pert"].torque.values == dname] for x in venus if x["angle"] is not None and len(x["pert"])])
    disp = np.nanmean(a - a[:, [za]], 0)
    print(f"{dname} torque: mean angle change at 0, 20, 50, 100 ms: "
          + ", ".join(f"{disp[za + t]:+.2f}" for t in [0, 20, 50, 100]) + " deg (positive = flexion)")

pr = pasq.pseudo_readings(venus, set(zip(studied.file, studied.unit)))
nt = pasq.no_torque_crossings(venus, set(zip(studied.file, studied.unit)))
print("\nany crossing of the claims file's threshold over 0 to 200 ms, per cell-direction:")
print("  after torque onset %.3f (n = %d), after pseudo onset %.3f, after the no-torque alignment %.3f (n = %d cells)"
      % (pr[pr.onset == "torque"].mean_sd.mean(), (pr.onset == "torque").sum(),
         pr[pr.onset == "pseudo"].mean_sd.mean(), nt.any_crossing.mean(), len(nt)))
on = np.concatenate([studied.fle_onset.dropna(), studied.ext_onset.dropna()])
print("response onsets after torque (ms): median %.0f, inside 20 to 60 by construction" % np.median(on))

The command leaves rest a few ms after the marked onset and the joint moves in the torque's
direction, so the analog series and the trials table share a clock (consistent with the data, as
far as a mean trace can show it). Spikes cross the threshold far more often after the real onset
than after the pseudo onset or in trials without a torque, so the spike times share it too.

Twelve files hold the trials of two lab source files, each with units the dataset merged across
them (named, for example, `v1601.1+v1601b.1`); no trial is repeated between the two, and no
source file appears in two NWB files. Each such unit is counted once, as the dataset gives it.

### Scale of the angle

The paper gives a mean torque-evoked displacement of 10 degrees; the files give about 3 (cell
`finding-kinematics-vs-paper`). The angle's scale is checked two ways: the voluntary movements of
the task, measured from the angle between the trials table's `derived_movement_onset_time` and
`derived_movement_end_time`, against their target distance and `derived_movement_amplitude`; and
the peak velocity of each perturbation computed from the angle against the peak of the file's own
`ElbowVelocity` series over the same 0 to 200 ms. The paper says monkey V's analog data were
digitized at 500 Hz; the files hold them at 1 kHz.

In [ ]:
vol = pd.concat([x["trials"].assign(post=bool(x["units"].post.iloc[0])) for x in venus
                 if "angle_at_mov_onset" in x["trials"]])
vol = vol.dropna(subset=["angle_at_mov_onset", "angle_at_mov_end", "derived_amplitude"])
vol["measured"] = (vol.angle_at_mov_end - vol.angle_at_mov_onset).abs()
vol["derived"] = vol.derived_amplitude.abs()
print(f"voluntary movements with an onset and end: {len(vol)}")
print(vol.groupby(["post", "target_amplitude"])[["measured", "derived"]].median().round(2)
      .join(vol.groupby(["post", "target_amplitude"]).size().rename("n")))
print("measured / derived amplitude: median %.3f, 5th to 95th percentile %.3f to %.3f"
      % tuple((vol.measured / vol.derived).quantile([0.5, 0.05, 0.95])))
k = kin_trials.dropna(subset=["file_peak_velocity", "peak_velocity"])
ratio = k.file_peak_velocity / k.peak_velocity
print("\nperturbations: peak of ElbowVelocity over 0 to 200 ms / peak velocity from the angle: median %.3f, "
      "5th to 95th percentile %.3f to %.3f (n = %d); Pearson r = %.3f"
      % (ratio.median(), ratio.quantile(0.05), ratio.quantile(0.95), len(k), np.corrcoef(k.file_peak_velocity, k.peak_velocity)[0, 1]))
print("peak velocity from the angle, range %.1f to %.1f deg/s; from ElbowVelocity %.1f to %.1f"
      % (k.peak_velocity.min(), k.peak_velocity.max(), k.file_peak_velocity.min(), k.file_peak_velocity.max()))
fig, ax = plt.subplots(figsize=(4.5, 4.5))
for post, color in [(False, PRE_COLOR), (True, POST_COLOR)]:
    g = k[k.post == post]
    ax.scatter(g.peak_velocity, g.file_peak_velocity, s=8, alpha=0.4, color=color, label="post-MPTP" if post else "pre-MPTP")
lim = [min(k.peak_velocity.min(), k.file_peak_velocity.min()), max(k.peak_velocity.max(), k.file_peak_velocity.max())]
ax.plot(lim, lim, color="0.5", lw=1)
ax.set_xlabel("peak velocity from ElbowAngle (deg/s)")
ax.set_ylabel("peak of ElbowVelocity (deg/s)")
ax.legend(frameon=False)
plt.show()

The angle's scale agrees with the task: voluntary movements to 20 degree targets measure 19.87
and 18.13 degrees from the angle (pre and post MPTP), and their ratio to `derived_movement_amplitude`
is 1.000. The peak of the file's `ElbowVelocity` is 1.028 times the peak velocity computed from the
angle (Pearson r = 1.000). The small torque-evoked displacements are therefore in the data, not a
scale error of this analysis.

### Findings about the data

Each cell below shows one problem, loading only what it needs.

In [ ]:
# Monkey L: no perturbation times, and no pulses in the torque command.
types = pd.Series(np.concatenate([x["torque_type"] for x in leu])).value_counts().to_dict()
print("Leu trials by torque_perturbation_type:", types)
finite = [x for x in leu if x["torque_cmd_finite"] > 0]
print("Leu files whose torque command has any finite value:", len(finite), "of", len(leu),
      "; of these after MPTP:", sum(bool(x["units"].post.iloc[0]) for x in finite))
steps = pd.Series({x["path"]: x["torque_cmd_steps"] for x in finite})
print("jumps of more than 20 units between successive samples of the command, per Leu file with a finite command:",
      sorted(steps.values))
# One command trace: the Leu file with the most jumps. Large excursions (above 100 units) are timed
# against its trials' events; Venus perturbations come 1.7 s after the centre target appears.
path = steps.idxmax()
with h5py.File(remfile.File(dict(leu_urls)[path]), "r") as h:
    tq = h["acquisition"][[k for k in h["acquisition"] if k.endswith("Torque")][0]]
    trace, t_tq = tq["data"][:], tq["timestamps"][:]
    tr = h["intervals/trials"]
    ltr = pd.DataFrame({k: tr[k][:] for k in ["start_time", "stop_time", "center_target_appearance_time",
                                              "derived_movement_onset_time"]})
print(path.split("/")[-1], f": {len(trace)} samples, median interval {np.median(np.diff(t_tq)) * 1000:.0f} ms, "
      f"range {np.nanmin(trace):.1f} to {np.nanmax(trace):.1f}")
big = np.abs(trace) > 100
ev = t_tq[np.where(big & ~np.r_[False, big[:-1]])[0]]
ev = ev[np.r_[True, np.diff(ev) > 0.3]]
k_ = np.searchsorted(ltr.start_time.values, ev) - 1
ev_df = pd.DataFrame({"after_center": ev - ltr.center_target_appearance_time.values[k_],
                      "after_movement_onset": ev - ltr.derived_movement_onset_time.values[k_]})
print(f"{len(ev)} excursions in {len(ltr)} trials; time after the centre target appears (s):",
      pd.cut(ev_df.after_center, [0, 0.1, 1, 1.5, 2.5, 3, 4, 5, 10]).value_counts(sort=False).to_dict())
print("time after the voluntary movement onset (s): median %.2f; within -0.05 to 0.15 s: %d"
      % (ev_df.after_movement_onset.median(), ev_df.after_movement_onset.between(-0.05, 0.15).sum()))
seg = t_tq <= ltr.stop_time.iloc[2]
fig, ax = plt.subplots(figsize=(8, 2.5))
ax.plot(t_tq[seg], trace[seg], color=PRE_COLOR, lw=1, label="torque command")
for t in ltr.center_target_appearance_time.iloc[:3]:
    ax.axvline(t, color="0.4", ls="--", lw=0.8)
for t in ltr.derived_movement_onset_time.iloc[:3]:
    ax.axvline(t, color=POST_COLOR, ls=":", lw=1)
ax.set_xlabel("time in file (s); dashed: centre target appears, dotted: movement onset")
ax.set_ylabel("command (a.u.)")
plt.show()


In [ ]:
# The torque command's pulse is about three times larger after MPTP; the paper gives one 0.1 Nm pulse.
print(kin_trials.groupby("post").command.describe().round(1))
print("\nsessions by median pulse size (a.u.), pre and post:")
print(pd.crosstab(kin_trials.groupby("file").post.first(),
                  pd.cut(kin_trials.groupby("file").command.median(), [0, 30, 60, 120, 250, 400])))

In [ ]:
# Torque-evoked displacements: the paper's mean is 10 degrees, and slower, smaller displacements after MPTP.
print(kin_sessions.groupby(["post", "direction"])[["peak_velocity", "amplitude", "latency"]].mean().round(2))
print("perturbations with displacement >= 10 deg: %.1f%%" % (100 * (kin_trials.amplitude >= 10).mean()))
print("sessions with mean peak velocity 130 to 200 deg/s:",
      int(kin_sessions.peak_velocity.between(130, 200).sum()), "of", len(kin_sessions))

## Claims

### cell-counts

> In monkey V, 112 neurons were studied before MPTP and 159 after, of which 49 and 54 were PTNs
> and 31 and 48 CSNs; three neurons in all were activated from both the peduncle and the putamen.

Criterion: counts of monkey V units with at least 5 perturbations per direction, by MPTP state and
projection type; dual-projecting units counted over all files.

In [ ]:
expected = {"M1": (112, 159), "PTN": (49, 54), "CSN": (31, 48)}
for g, m in groups.items():
    n = studied[m].groupby("post").size()
    print(f"{g}: pre {n.get(False, 0)} (expected {expected[g][0]}), post {n.get(True, 0)} (expected {expected[g][1]})")
dual = all_units[all_units.ptype == "pyramidal_tract_and_corticostriatal_neuron"]
print("activated from both peduncle and putamen, all files:", len(dual), dual.groupby("subject").size().to_dict(),
      "; of these studied:", int((studied.ptype == "pyramidal_tract_and_corticostriatal_neuron").sum()))
print("other types among the studied cells:", studied[~studied.ptype.isin([PTN, CSN])].groupby(["post", "ptype"]).size().to_dict())
print("Venus units left out for fewer than 5 perturbations in a direction:",
      cells[~cells.studied].groupby("post").size().to_dict())
# The claims file's data check found 112 and 159 cells. Counts under two other readings of "at least
# 5 perturbations of each direction": all of the file's perturbations, whatever the unit's
# observation intervals; and observation required only over -500 to +200 ms (here: -1500 to +500).
alt = []
for fi, x in enumerate(venus):
    p = x["pert"]
    for ui, u in x["units"].iterrows():
        ob = x["obs"][ui]
        n_obs = [sum(np.any((ob[:, 0] <= t - 0.5) & (ob[:, 1] >= t + 0.2)) for t in p.onset[p.torque == d])
                 for d in pasq.DIRECTIONS]
        alt.append({"post": u.post, "ptype": u.ptype, "file": min((p.torque == d).sum() for d in pasq.DIRECTIONS),
                    "obs_500_200": min(n_obs)})
alt = pd.DataFrame(alt)
for col in ["file", "obs_500_200"]:
    s = alt[alt[col] >= pasq.MIN_PERTURBATIONS]
    print(f"rule {col}: M1 {s.groupby('post').size().to_dict()}, PTN {s[s.ptype == PTN].groupby('post').size().to_dict()}, "
          f"CSN {s[s.ptype == CSN].groupby('post').size().to_dict()}")

**Result: reproduced.** 115 and 157 cells before and after MPTP (112 and 159 expected), 51 and 54
PTNs (49 and 54), 32 and 48 CSNs (31 and 48), and 3 dual-projecting cells, all within tolerance. Counting all of a file's perturbations, or requiring the unit's observation only over -500 to
+200 ms, gives the same counts, so the difference from the claims file's data check (112 and 159)
is not explained by the observation rule.

### monkey-l-perturbations

> In monkey L, 57 of 115 neurons before MPTP and 35 of 73 after responded to torque perturbations
> of the wrist, and the monkey L responses enter all pooled results.

Blocked: no Leu trial marks a perturbation (cell `finding-leu-no-perturbations`), and the torque
command is NaN in most files.

In [ ]:
print("Leu units by state:", all_units[all_units.subject == "Leu"].groupby("post").size().to_dict())
print("Leu trials with a perturbation onset:", int(sum(np.isfinite(x["onset"]).sum() for x in leu)))

**Result: blocked (data missing).** The Leu files hold 138 and 72 units but no perturbation
times, and the torque command is NaN in 114 of 148 files. In the post-MPTP file with the most
jumps in its command, the 303 large excursions fall at the appearance of the centre target (120,
one in every trial, within 0.1 s) or 2.5 to 10 s after it, none 1 to 2.5 s after it where the
Venus perturbations fall (1.7 s); 55 fall within 0.15 s of a voluntary movement onset. They are
timed to the task, not to perturbations, so the command does not supply the missing times.

### baseline-rate

> Before torque onset, the firing rate of M1 neurons was lower after MPTP, because PTN rates fell
> (Mann-Whitney p < 0.001) while CSN rates did not change (p = 0.5). In monkey V the rates were 12.6
> and 8.9 spikes/s for all cells, 20.3 and 14.8 for PTNs, and 3.5 and 3.4 for CSNs.

Baseline rate: the mean over all perturbations of the SDF over the 500 ms before onset (claims).

In [ ]:
exp = {"M1": (12.6, 1.98, 8.9, 1.33), "PTN": (20.3, 2.46, 14.8, 2.72), "CSN": (3.5, 1.47, 3.4, 1.01)}
for g, m in groups.items():
    s = studied[m]
    a, b = s[~s.post].baseline_rate, s[s.post].baseline_rate
    p = stats.mannwhitneyu(a, b, alternative="two-sided").pvalue
    e = exp[g]
    print(f"{g}: pre {a.mean():.2f} +/- {a.std():.2f} (expected {e[0]} +/- {e[1]}, n {len(a)}), "
          f"post {b.mean():.2f} +/- {b.std():.2f} (expected {e[2]} +/- {e[3]}, n {len(b)}); "
          f"change {100 * (b.mean() / a.mean() - 1):+.0f}%, Mann-Whitney p = {p:.3g}")

**Result: reproduced.** All six means are within tolerance (M1 12.66 and 8.85, PTN 18.49 and
14.37, CSN 3.48 and 3.36 spikes/s); the PTN rate is lower after MPTP (Mann-Whitney p = 0.020) and
the CSN rate does not differ (p = 0.65). The PTN test does not reach the paper's p < 0.001; the M1
rate is also lower (p = 0.0022, below the paper's 0.05/2).

### ptn-faster-than-csn

> Before MPTP, PTNs fired much faster than CSNs at rest (Mann-Whitney p < 0.001).

In [ ]:
pre = studied[~studied.post]
a, b = pre[pre.ptype == PTN].baseline_rate, pre[pre.ptype == CSN].baseline_rate
print(f"pre-MPTP PTN {a.mean():.2f} spikes/s (range {a.min():.1f} to {a.max():.1f}, n {len(a)}), "
      f"CSN {b.mean():.2f} (range {b.min():.1f} to {b.max():.1f}, n {len(b)}); "
      f"Mann-Whitney p = {stats.mannwhitneyu(a, b, alternative='two-sided').pvalue:.2g}")
print("reported, not judged: text 16.3 and 3.6, Table 1 18.3 and 3.2 spikes/s (both monkeys)")

**Result: reproduced.** PTNs 18.49 against CSNs 3.48 spikes/s, Mann-Whitney p far below 0.001.

### antidromic-latency

> PTN antidromic latencies did not change after MPTP (1.9 ms, p = 0.9), while CSN latencies were
> longer (5.1 against 6.1 ms, p < 0.05/3).

In [ ]:
exp = {"PTN": (1.9, 0.22, 1.9, 0.2), "CSN": (5.1, 0.61, 6.1, 0.89)}
leu_units = all_units[all_units.subject == "Leu"]
for label, data in [("monkey V studied", studied), ("variant, + every Leu unit", None)]:
    for g, t in [("PTN", PTN), ("CSN", CSN)]:
        s = studied[studied.ptype == t][["post", "latency"]]
        if data is None:
            s = pd.concat([s, leu_units[leu_units.ptype == t][["post", "latency"]]])
        a, b = s[~s.post].latency.dropna(), s[s.post].latency.dropna()
        p = stats.mannwhitneyu(a, b, alternative="two-sided").pvalue
        e = exp[g]
        print(f"{label} {g}: pre {a.mean():.2f} ms (expected {e[0]} +/- {e[1]}, n {len(a)}), "
              f"post {b.mean():.2f} (expected {e[2]} +/- {e[3]}, n {len(b)}), Mann-Whitney p = {p:.3g}")

**Result: partially reproduced.** All four means are within tolerance (PTN 1.92 and 1.86 ms, CSN
4.91 and 6.85 ms) and the PTN latency does not differ (p = 0.68), but the CSN latency, longer after
MPTP, has p = 0.019, above the 0.05/3 the criterion asks. With every Leu unit added (variant), the
CSN test gives p = 0.0072.

### false-positive-rate

> The detection threshold, p < 0.01 divided by 5 independent comparisons, holds the chance of a
> false response over the 200 ms epoch near 0.01.

Criterion: the same detection at a pseudo onset 600 ms before the torque, on the studied cells,
both directions; reproduced when the fraction of cell-directions with any threshold crossing over
0 to 200 ms is between 0.0033 and 0.03. The other readings of the Methods' test (step 1.6) are
shown with the fraction of real cell-directions each would call responsive.

In [ ]:
pse = pr[pr.onset == "pseudo"]
print("claims file's detection, pseudo onset: any crossing over 0 to 200 ms %.3f, response with onset in 20 to 60 ms %.3f (n = %d cell-directions)"
      % (pse.mean_sd.mean(), np.r_[studied.pse_fle_responds, studied.pse_ext_responds].mean(), len(pse)))
print("no-torque trials, same detection: any crossing %.3f, response 20 to 60 ms %.3f (n = %d cells)"
      % (nt.any_crossing.mean(), nt.responds.mean(), len(nt)))
print("\nreadings of the test (fraction of cell-directions with any crossing over 0 to 200 ms):")
print(pr.groupby("onset")[["mean_sd", "ttest", "trial_se", "trial_sd"]].mean().round(3).T)
for label, s in [("pre", studied[~studied.post]), ("post", studied[studied.post])]:
    print(f"pseudo any-crossing rate {label}-MPTP: "
          f"{np.r_[s.pse_fle_any_crossing, s.pse_ext_any_crossing].mean():.3f}")

**Result: blocked (methods underspecified).** The claims file's detection crosses its threshold in
0.23 of cell-directions after a pseudo onset (0.20 in trials without a torque), far above the
paper's 0.01 and outside 0.0033 to 0.03, so this procedure is not the paper's. Of the other
readings, only a threshold from the SD of all single-trial baseline values comes near 0.01
(0.013), and it calls only 0.20 of real cell-directions responsive, far below the paper's 62 to
75% of cells; no reading gives both. The question of
which test the Methods mean is asked of a reader (`questions/pasquereau-2013.yaml`) and of the
authors (findings).

### Claims that use the selection of responses (blocked, shown)

These claims depend on `torque-related-fractions`, which depends on `false-positive-rate`, and are
blocked with it. Their results under the claims file's detection are computed here so that a later
attempt, or the authors, can see them; they decide nothing.

Each cell also gives the results under the `trial_sd` reading (threshold from the SD of all
single-trial SDF values over the baseline), the one reading of the Methods' t-test whose
false-positive rate (0.013) falls inside 0.0033 to 0.03 (cell `claim:false-positive-rate`). It
is not the claims file's detection, so it decides nothing either; it is shown so that the
dependence of these claims on the reading can be seen. A question on the detection is asked of a
reader (`questions/pasquereau-2013.yaml`).

In [ ]:
def as_reading(df, prefix):
    """The studied cells with one reading's response columns under the names fle_* and ext_*."""
    out = df.copy()
    for c in [c for c in df.columns if c.startswith(prefix)]:
        out[c[len(prefix):]] = df[c]
    return out

readings = {"claims file": studied, "trial_sd": as_reading(studied, "tsd_")}
for name, st in readings.items():
    st["tr"] = st.fle_responds.fillna(False).astype(bool) | st.ext_responds.fillna(False).astype(bool)

In [ ]:
exp = {"M1": (75, 68), "PTN": (55, 68), "CSN": (45, 21)}
for name, st in readings.items():
    print(f"reading: {name}")
    for g, m in groups.items():
        s = st[m]
        f = s.groupby("post").tr.agg(["sum", "size", "mean"])
        print(f"  {g}: pre {int(f.loc[False, 'sum'])}/{int(f.loc[False, 'size'])} ({100 * f.loc[False, 'mean']:.1f}%, expected {exp[g][0]}), "
              f"post {int(f.loc[True, 'sum'])}/{int(f.loc[True, 'size'])} ({100 * f.loc[True, 'mean']:.1f}%, expected {exp[g][1]})")
studied["tr_ttest"] = studied.fle_ttest_responds.fillna(False).astype(bool) | studied.ext_ttest_responds.fillna(False).astype(bool)
print("variant (t-test of single trials):", (100 * studied.groupby("post").tr_ttest.mean()).round(1).to_dict())

In [ ]:
def yates(table):
    chi2, p, _, _ = stats.chi2_contingency(np.asarray(table), correction=True)
    return chi2, p

for name, st in readings.items():
    print(f"reading: {name}")
    for g, m in groups.items():
        s = st[m]
        chi2, p = yates(pd.crosstab(s.post, s.tr))
        print(f"  {g}: torque-related pre {100 * s[~s.post].tr.mean():.1f}%, post {100 * s[s.post].tr.mean():.1f}%, "
              f"Yates chi2 = {chi2:.2f}, p = {p:.3g}")

In [ ]:
for name, st in readings.items():
    s = st[st.ptype.isin([PTN, CSN])]
    chi2, p = yates(pd.crosstab(s.ptype, s.tr))
    print(f"reading {name}: PTN {100 * s[s.ptype == PTN].tr.mean():.1f}% (expected 62.1), CSN {100 * s[s.ptype == CSN].tr.mean():.1f}% (expected 30.4); "
          f"Yates chi2 = {chi2:.1f}, p = {p:.2g}")

In [ ]:
for name, st in readings.items():
    t_ = st[st.tr]
    one = t_.fle_responds.fillna(False).astype(bool) ^ t_.ext_responds.fillna(False).astype(bool)
    print(f"reading {name}: one direction only {one.sum()}/{len(t_)} ({100 * one.mean():.1f}%, expected 44)")

In [ ]:
def make_resp(st):
    rows = []
    for _, c in st.iterrows():
        for d in ["fle", "ext"]:
            if c.get(f"{d}_responds") is True:
                rows.append({"file": c.file, "unit": c.unit, "post": c.post, "ptype": c.ptype, "direction": d,
                             "date": c.date, "baseline_rate": c.baseline_rate,
                             **{k: c[f"{d}_{k}"] for k in ["onset", "offset", "sign", "mean_change", "max_change",
                                                             "area", "mean_rate", "fwhm"]}})
    r = pd.DataFrame(rows)
    r["increase"] = r.sign > 0
    return r

RESP = {name: make_resp(st) for name, st in readings.items()}
resp = RESP["claims file"]
for name, r in RESP.items():
    print(f"reading {name}: {len(r)} responses in {r.groupby(['file', 'unit']).ngroups} cells")
    for g, sel in [("M1", r.index == r.index), ("PTN", r.ptype == PTN)]:
        s = r[sel]
        print(f"  {g}: increases pre {100 * s[~s.post].increase.mean():.1f}%, post {100 * s[s.post].increase.mean():.1f}%")
    s = r[r.ptype == PTN]
    chi2, p = yates(pd.crosstab(s.post, s.increase))
    print(f"  PTN increases against decreases, pre against post: Yates chi2 = {chi2:.2f}, p = {p:.3g}")
print("expected: M1 86 and 88, PTN 70 and 75")

In [ ]:
for name, r in RESP.items():
    print(f"reading: {name}")
    for g, sel in [("M1", r.index == r.index), ("PTN", r.ptype == PTN)]:
        s = r[sel]
        for meas in ["mean_change", "max_change", "area"]:
            a, b = s[~s.post][meas], s[s.post][meas]
            print(f"  {g} {meas}: pre median {a.median():.2f} (n {len(a)}), post {b.median():.2f} (n {len(b)}); "
                  f"KS p = {stats.ks_2samp(a, b).pvalue:.3g}, Mann-Whitney p = {stats.mannwhitneyu(a, b, alternative='two-sided').pvalue:.3g}")

In [ ]:
exp = {("M1", "fle"): (42.4, 37.9), ("M1", "ext"): (50.9, 42.6), ("PTN", "fle"): (33.4, 26.1), ("PTN", "ext"): (38.8, 29.3)}
for name, r in RESP.items():
    print(f"reading: {name}")
    for reading in ["max_change", "mean_rate", "mean_change"]:
        out = []
        for (g, d), e in exp.items():
            s = r[(r.direction == d) & ((r.ptype == PTN) if g == "PTN" else True)]
            out.append(f"{g} {d} {s[~s.post][reading].mean():.1f}/{s[s.post][reading].mean():.1f} (exp {e[0]}/{e[1]})")
        print(f"  {reading}: " + "; ".join(out))

In [ ]:
exp = {("M1", "fle"): (39.2, 34.56), ("M1", "ext"): (40.4, 33.4), ("PTN", "fle"): (43.9, 35.5), ("PTN", "ext"): (44.4, 36.7)}
EARLIEST = {}
for name, r in RESP.items():
    print(f"reading: {name}")
    for (g, d), e in exp.items():
        s = r[(r.direction == d) & ((r.ptype == PTN) if g == "PTN" else True)]
        a, b = s[~s.post].onset, s[s.post].onset
        print(f"  {g} {d}: pre {a.mean():.1f} ms (exp {e[0]}, n {len(a)}), post {b.mean():.1f} (exp {e[1]}, n {len(b)}), "
              f"KS p = {stats.ks_2samp(a, b).pvalue:.3g}")
    EARLIEST[name] = r.groupby(["file", "unit", "post", "ptype", "date"]).onset.min().reset_index()
    for g, s in [("M1", EARLIEST[name]), ("PTN", EARLIEST[name][EARLIEST[name].ptype == PTN])]:
        print(f"  {g} earliest per cell: pre {s[~s.post].onset.mean():.1f}, post {s[s.post].onset.mean():.1f} ms")
earliest = EARLIEST["claims file"]

In [ ]:
for name, e_ in EARLIEST.items():
    e_["day"] = (pd.to_datetime(e_.date) - first_day).dt.days
    rho, p = stats.spearmanr(e_.day, e_.onset)
    print(f"reading {name}: latency against days since the first Venus session: Spearman rho = {rho:.3f}, p = {p:.3g}, n = {len(e_)}")
    for post, s in e_.groupby("post"):
        r_, pp = stats.spearmanr(s.day, s.onset)
        print(f"  {'post' if post else 'pre'} only: rho = {r_:.3f}, p = {pp:.3g}, n = {len(s)}")

In [ ]:
exp = {"fle": (97, 52), "ext": (57, 41)}
for name, r in RESP.items():
    print(f"reading: {name}")
    for d, e in exp.items():
        s = r[(r.ptype == PTN) & (r.direction == d)]
        inc = s[s.increase]
        a, b = inc[~inc.post].fwhm, inc[inc.post].fwhm
        print(f"  PTN {d} increases: FWHM pre {a.mean():.1f} ms (exp {e[0]}, n {len(a)}), post {b.mean():.1f} (exp {e[1]}, n {len(b)}), "
              f"Mann-Whitney p = {stats.mannwhitneyu(a, b, alternative='two-sided').pvalue:.3g}; all responses "
              f"{s[~s.post].fwhm.mean():.1f} and {s[s.post].fwhm.mean():.1f}")

In [ ]:
def dsi(c):
    f, e = c.fle_max_change_signed, c.ext_max_change_signed
    p_, np_ = (f, e) if abs(f) >= abs(e) else (e, f)
    return abs(1 - np_ / p_)

def category(c):
    fr, er = c.fle_responds is True, c.ext_responds is True
    if fr != er:
        return "unidirectional"
    if c.fle_sign != c.ext_sign:
        return "reciprocal"
    return "bidirectional" if c.dsi > 0.5 else "non-directional"

TR = {}
for name, st in readings.items():
    t_ = st[st.tr].copy()
    t_["dsi"] = t_.apply(dsi, axis=1)
    t_["category"] = t_.apply(category, axis=1)
    TR[name] = t_
    ct = pd.crosstab(t_.post, t_.category)
    print(f"reading: {name}")
    print(ct)
    chi2, p, _, _ = stats.chi2_contingency(ct)
    hi = t_.category.isin(["reciprocal", "unidirectional"])
    print(f"reciprocal + unidirectional: pre {100 * hi[~t_.post].mean():.1f}%, post {100 * hi[t_.post].mean():.1f}%; "
          f"chi2 = {chi2:.1f}, p = {p:.3g}\n")

In [ ]:
exp = {"M1": (0.72, 0.66), "PTN": (0.73, 0.75), "CSN": (0.81, 0.83)}
for name, t_ in TR.items():
    print(f"reading: {name}")
    for g, e in exp.items():
        s = t_ if g == "M1" else t_[t_.ptype == (PTN if g == "PTN" else CSN)]
        print(f"  {g}: DSI pre {s[~s.post].dsi.mean():.2f} (exp {e[0]}, n {(~s.post).sum()}), "
              f"post {s[s.post].dsi.mean():.2f} (exp {e[1]}, n {s.post.sum()})")

In [ ]:
for name, t_ in TR.items():
    F, p = stats.f_oneway(*[g.baseline_rate.values for _, g in t_.groupby("category")])
    c3 = t_.category.replace({"reciprocal": "high", "unidirectional": "high"})
    F3, p3 = stats.f_oneway(*[g.baseline_rate.values for _, g in t_.groupby(c3)])
    print(f"reading {name}: baseline rate across {t_.category.nunique()} categories: F = {F:.2f}, p = {p:.3g}; "
          f"variant, three groups: F = {F3:.2f}, p = {p3:.3g}")

In [ ]:
vel = kin_sessions.set_index(["file", "direction"]).peak_velocity
for name, r in RESP.items():
    for d, dn in [("fle", "flexion"), ("ext", "extension")]:
        s = r[r.direction == d]
        v = np.array([vel.get((f, dn), np.nan) for f in s.file])
        s = s[(v >= 130) & (v <= 200)]
        print(f"reading {name}, {dn}: responses from sessions with mean peak velocity 130 to 200 deg/s: pre {(~s.post).sum()}, post {s.post.sum()}")

In [ ]:
emg = pasq.session_emg(venus)
v_both = kin_sessions.groupby("file").peak_velocity.mean()
m2_tl1 = emg[(emg.channel == "EMGTricepsLongus1") & ~emg.has_nan & (emg.baseline_sd >= 1)].set_index("file").m2
for name, e_ in EARLIEST.items():
    ptn_lat = e_[e_.ptype == PTN].groupby("file").onset.mean()
    for label, other in [("peak velocity", v_both), ("triceps longus 1 m2", m2_tl1)]:
        j = pd.concat([ptn_lat, other], axis=1, join="inner").dropna()
        rho, p = stats.spearmanr(j.iloc[:, 0], j.iloc[:, 1])
        print(f"reading {name}: mean PTN latency against {label}: rho = {rho:.3f}, p = {p:.3g}, n = {len(j)} sessions")

**Result: blocked (methods underspecified), through false-positive-rate.** The numbers above
decide nothing. The claims file's detection has a false-positive rate of 0.23 per cell-direction,
about 20 times the paper's. Under the `trial_sd` reading, whose false-positive rate is 0.013, only
20.0% and 18.5% of M1 cells are torque-related before and after MPTP (75 and 68% expected), 19.6%
and 14.8% of PTNs and 3.1% and 0.0% of CSNs, and every response it finds is an increase, so the
selection it makes is far from the paper's and would not reproduce torque-related-fractions
either. No session has a mean peak velocity in the 130 to 200 deg/s range of
velocity-matched-magnitude under either reading.

### kinematics-slower

> After MPTP, torque-induced displacements were slower and smaller; mean peak velocity fell by
> 16.2% and amplitude by 6.6% (ANOVA, F > 515 and F > 49).

Criterion: session means per direction, monkey V; two-way ANOVA (type II) of MPTP state and
direction with their interaction; the change is the post mean over the pre mean, minus 1;
readings (a) both directions pooled, (b) flexion only. Kinematic parameters: `pasq.py` docstring.

In [ ]:
def anova(df, meas):
    model = smf.ols(f"{meas} ~ C(post) * C(direction)", data=df).fit()
    return sm.stats.anova_lm(model, typ=2)

print("sessions:", kin_sessions.groupby(["post", "direction"]).size().to_dict())
print("perturbations without a movement onset within 200 ms:", int(kin_trials.latency.isna().sum()), "of", len(kin_trials))
for meas in ["peak_velocity", "amplitude"]:
    a = anova(kin_sessions, meas)
    pooled = 100 * (kin_sessions[kin_sessions.post][meas].mean() / kin_sessions[~kin_sessions.post][meas].mean() - 1)
    f = kin_sessions[kin_sessions.direction == "flexion"]
    flex = 100 * (f[f.post][meas].mean() / f[~f.post][meas].mean() - 1)
    print(f"\n{meas}: change (a) pooled {pooled:+.1f}%, (b) flexion {flex:+.1f}%; MPTP F = {a.loc['C(post)', 'F']:.1f}, "
          f"p = {a.loc['C(post)', 'PR(>F)']:.2g}")
    print(a.round(4))

A variant, not deciding: the pulse of the torque command is about three times larger after MPTP
(cell `finding-torque-command-levels`). Restricting to perturbations whose command pulse is 60 to
250 units, the range both states share, gives the comparison at similar torques.

In [ ]:
matched = kin_trials[kin_trials.command.between(60, 250)]
ms = matched.groupby(["file", "post", "direction"])[["latency", "peak_velocity", "amplitude"]].mean().reset_index()
print("perturbations kept:", matched.groupby("post").size().to_dict(), "; sessions:", ms.groupby("post").size().to_dict())
for meas in ["peak_velocity", "amplitude", "latency"]:
    a = anova(ms, meas)
    ch = 100 * (ms[ms.post][meas].mean() / ms[~ms.post][meas].mean() - 1)
    print(f"{meas}: pre {ms[~ms.post][meas].mean():.2f}, post {ms[ms.post][meas].mean():.2f}, change {ch:+.1f}%, "
          f"MPTP F = {a.loc['C(post)', 'F']:.1f}, p = {a.loc['C(post)', 'PR(>F)']:.2g}")

**Result: not reproduced.** Under both readings the displacements are faster and larger after
MPTP, not slower and smaller (peak velocity +28.1% pooled and +29.7% for flexion; amplitude +14.8%
and +11.1%; MPTP F = 98.5 and 25.0). Restricted to command pulses of 60 to 250 units the increase
remains (+47.0% and +40.4%), so the command's arbitrary units do not explain it.

### kinematic-latency-unchanged

> The latency of torque-induced displacements did not change after MPTP (F < 0.2, p > 0.05).

Criterion: the MPTP effect in the two-way ANOVA of session mean onset latency. The claims file's
filter is applied forwards and backwards, so it spreads the velocity's rise to earlier times; the
search from 0 ms does not ask that the velocity be below threshold before onset. Two variants
(not deciding) check what this does: the same filter with the velocity required to be at or below
5 deg/s over the 20 ms before the onset (`latency_quiet`), and a causal 25 Hz filter with the same
requirement (`latency_causal`, `pasq.py`). The unfiltered velocity shows when the joint starts to
move.

In [ ]:
a = anova(kin_sessions, "latency")
print(kin_sessions.groupby(["post", "direction"]).latency.mean().round(2))
print(a.round(4))
print("\nper perturbation, claims file's onset:")
for post, g in kin_trials.groupby("post"):
    print(f"  {'post' if post else 'pre'}: onset at 0 ms {(g.latency == 0).mean():.3f}; velocity at torque onset median "
          f"{g.velocity_at_0.median():.2f} deg/s, above 5 deg/s {(g.velocity_at_0 > 5).mean():.3f}; "
          f"onset 10, 25, 50, 75, 90th percentile {g.latency.quantile([.1, .25, .5, .75, .9]).round(1).tolist()} ms")
za = -pasq.ANGLE_WINDOW_MS[0]
for post in [False, True]:
    v = np.vstack([np.where(x["pert"].torque.values == "flexion", 1.0, -1.0)[:, None] * np.gradient(x["angle"], axis=1) * 1000
                   for x in venus if x["angle"] is not None and len(x["pert"]) and bool(x["units"].post.iloc[0]) == post])
    print(f"  {'post' if post else 'pre'}: mean unfiltered velocity at 0, 4, 8, 10, 12, 14, 16, 20 ms: "
          + ", ".join(f"{np.nanmean(v[:, za + t]):.1f}" for t in [0, 4, 8, 10, 12, 14, 16, 20]) + " deg/s")
print("\nvariants, session means:")
for meas in ["latency_quiet", "latency_causal"]:
    for label, kt in [("all perturbations", kin_trials), ("command 60 to 250 units", kin_trials[kin_trials.command.between(60, 250)])]:
        s = kt.groupby(["file", "post", "direction"])[meas].mean().reset_index().dropna()
        a = anova(s, meas)
        print(f"{meas}, {label}: perturbations left out {kt[meas].isna().groupby(kt.post).mean().round(3).to_dict()}; "
              f"means {s.groupby(['post', 'direction'])[meas].mean().round(2).to_dict()}; "
              f"MPTP F = {a.loc['C(post)', 'F']:.1f}, p = {a.loc['C(post)', 'PR(>F)']:.2g}; "
              f"interaction F = {a.loc['C(post):C(direction)', 'F']:.1f}")

**Result: not reproduced.** The movement onset latency is shorter after MPTP, MPTP effect F = 709,
p < 1e-100; the MPTP by direction interaction is F = 16.1 (the paper's F > 94). The claims file's
filter does move onsets early after MPTP: 6.7% of post-MPTP onsets are at 0 ms with the velocity
already above 5 deg/s (0.1% before). The variants that avoid this still find shorter latencies
after MPTP: with the velocity required to be below threshold beforehand (pre 10.87 and 9.65 ms,
post 6.13 and 3.38 ms for extension and flexion; F = 596.5) and with a causal filter (pre 21.44
and 20.33, post 18.18 and 15.62 ms; F = 422.5), and both again at command pulses of 60 to 250
units (F = 214.3 and 146.2). The unfiltered velocity also rises earlier after MPTP (5.1 against
1.7 deg/s at 10 ms, 13.2 against 4.0 at 12 ms).

### EMG claims

Session-average EMG of the three triceps channels for flexion torques (claims). A channel is used
in a session when no window holds a NaN and its baseline SD is at least 1 A/D unit; the claims
file does not say whether this is the SD of the session-average trace (main, the SD the response
threshold uses) or of the single-trial samples (shown as a variant).

In [ ]:
def emg_used(rule):
    return emg[~emg.has_nan & (emg[rule] >= 1)]

print("channel-sessions:", emg.groupby(["channel", "post"]).size().to_dict())
print("with NaN in a window:", int(emg.has_nan.sum()))
for rule in ["baseline_sd", "raw_sd"]:
    print(f"used under {rule} >= 1:", emg_used(rule).groupby(["channel", "post"]).size().to_dict())

In [ ]:
# Background of the signals: mean raw value and SD over the 200 ms baseline, by channel and state
print(emg[~emg.has_nan].groupby(["channel", "post"])[["level", "raw_sd", "baseline_sd"]].median().round(2))
print("\nfraction of channel-sessions whose session-average baseline SD is below 1 unit:")
print(emg[~emg.has_nan].groupby(["channel", "post"]).baseline_sd.apply(lambda s: (s < 1).mean()).round(2).to_dict())
# Grand averages of the session-average traces, less their baseline, for flexion torques
ze = -pasq.EMG_WINDOW_MS[0]
traces = {}
for r in emg_used("baseline_sd").itertuples():
    x = venus[r.file]
    w = x["emg"][r.channel][x["pert"].torque.values == "flexion"].mean(0)
    traces.setdefault((r.channel, r.post), []).append(w - w[ze - pasq.EMG_BASELINE_MS:ze].mean())
ts_ms = np.arange(pasq.EMG_WINDOW_MS[0], pasq.EMG_WINDOW_MS[1])
fig, axes = plt.subplots(1, 3, figsize=(12, 3), sharey=False)
for ax, ch in zip(axes, pasq.TRICEPS):
    for post, color in [(False, PRE_COLOR), (True, POST_COLOR)]:
        g = np.mean(traces[(ch, post)], 0)
        print(f"{ch} {'post' if post else 'pre'} (n {len(traces[(ch, post)])}): grand average at 0, 20, 30, 50, 70, 100, 150, 200 ms: "
              + ", ".join(f"{g[ze + t]:.1f}" for t in [0, 20, 30, 50, 70, 100, 150, 200])
              + f"; range over -50 to 250 ms {g[ze - 50:ze + 250].min():.1f} to {g[ze - 50:ze + 250].max():.1f}")
        ax.plot(ts_ms[ze - 50:ze + 250], g[ze - 50:ze + 250], color=color, lw=1.5, label="post-MPTP" if post else "pre-MPTP")
    ax.set_title(ch, fontsize=10)
    ax.set_xlabel("ms from torque onset")
axes[0].set_ylabel("EMG less baseline (a.u.)")
axes[0].legend(frameon=False)
plt.show()

### EMG claims at matched torque commands (variant)

The command pulse is about three times larger after MPTP (cell `finding-torque-command-levels`).
The four EMG measures are computed again on the perturbations whose pulse is 60 to 250 units,
the range both states share, with the main channel rule. Not deciding.

In [ ]:
emg_m = pasq.session_emg(venus, command_range=(60, 250))
um = emg_m[~emg_m.has_nan & (emg_m.baseline_sd >= 1)]
fl = kin_trials[(kin_trials.direction == "flexion") & kin_trials.command.between(60, 250)]
print("flexion perturbations kept:", fl.groupby("post").size().to_dict(), "; their median command",
      fl.groupby("post").command.median().round(1).to_dict())
print("channel-sessions used:", um.groupby(["channel", "post"]).size().to_dict())
for ch, g in um.groupby("channel"):
    a, b = g[~g.post].m2, g[g.post].m2
    print(f"  m2 {ch}: median pre {a.median():.1f}, post {b.median():.1f}, KS p = {stats.ks_2samp(a, b).pvalue:.2g}")
d = um.dropna(subset=["duration"])
a, b = d[~d.post], d[d.post]
print(f"duration pre {a.duration.mean():.1f} ms (n {len(a)}), post {b.duration.mean():.1f} (n {len(b)}), KS p = {stats.ks_2samp(a.duration, b.duration).pvalue:.2g}")
print(f"onset pre {a.onset.mean():.1f} ms, post {b.onset.mean():.1f}, KS p = {stats.ks_2samp(a.onset, b.onset).pvalue:.2g}")
for post, g in um.groupby("post"):
    print(f"{'post' if post else 'pre'}: m2 > m1 in {100 * (g.m2 > g.m1).mean():.1f}% of {len(g)}, median m1/m2 {(g.m1 / g.m2).median():.2f}")

In [ ]:
for rule in ["baseline_sd", "raw_sd"]:
    print("rule", rule)
    for ch, g in emg_used(rule).groupby("channel"):
        a, b = g[~g.post].m2, g[g.post].m2
        print(f"  {ch}: m2 median pre {a.median():.1f}, post {b.median():.1f} (a.u.), n {len(a)} and {len(b)}, "
              f"KS p = {stats.ks_2samp(a, b).pvalue:.2g}")

**Result: reproduced.** m2 is larger after MPTP in all three triceps channels (KS p < 1e-14 in
each) under both channel rules. The torque pulses are about three times larger after MPTP (cell
`finding-torque-command-levels`); at pulses of 60 to 250 units (variant above; median 90.9 and
181.8 units) m2 is still larger after MPTP in every channel (KS p 7.5e-08 to 2.2e-15), though the
pulses still differ by a factor of two. The background also differs between the states: the
median single-trial baseline SD of triceps longus 2 is 2.98 units before and 19.58 after MPTP,
and half of its pre-MPTP session averages have a baseline SD below 1 unit.

In [ ]:
for rule in ["baseline_sd", "raw_sd"]:
    u = emg_used(rule)
    d = u.dropna(subset=["duration"])
    a, b = d[~d.post].duration, d[d.post].duration
    print(f"rule {rule}: duration pre {a.mean():.1f} ms (exp 57 +/- 6, n {len(a)}), post {b.mean():.1f} (exp 79 +/- 3, n {len(b)}), "
          f"KS p = {stats.ks_2samp(a, b).pvalue:.2g}; channel-sessions without a response: "
          f"{u.duration.isna().groupby(u.post).sum().to_dict()}")
    print("  by channel:", d.groupby(["channel", "post"]).duration.mean().round(1).to_dict())

**Result: partially reproduced.** Durations are longer after MPTP (22.2 against 51.5 ms, KS
p < 1e-30), but both means are far outside the paper's 57 +/- 6 and 79 +/- 3 ms. At matched command pulses (variant above) the
durations are 32.0 and 51.1 ms, still outside both tolerances.

In [ ]:
for rule in ["baseline_sd", "raw_sd"]:
    d = emg_used(rule).dropna(subset=["duration"])
    a, b = d[~d.post].onset, d[d.post].onset
    print(f"rule {rule}: onset pre {a.mean():.1f} ms (median {a.median():.0f}), post {b.mean():.1f} (median {b.median():.0f}), "
          f"KS p = {stats.ks_2samp(a, b).pvalue:.2g}")

**Result: not reproduced.** The earliest EMG responses start earlier after MPTP (44.2 against 25.0
ms on average, KS p < 1e-20). At matched command pulses (variant above) the onset is
also earlier after MPTP (33.3 against 24.7 ms, KS p = 8.5e-07).

In [ ]:
for rule in ["baseline_sd", "raw_sd"]:
    u = emg_used(rule)
    for post, g in u.groupby("post"):
        print(f"rule {rule}, {'post' if post else 'pre'}: m2 > m1 in {(g.m2 > g.m1).sum()}/{len(g)} "
              f"({100 * (g.m2 > g.m1).mean():.1f}%), median m1/m2 {(g.m1 / g.m2).median():.2f}")

**Result: partially reproduced.** After MPTP m2 exceeds m1 in 98.3% of channel-sessions and the
median m1/m2 is 0.25; before MPTP the median ratio is 0.14 but m2 exceeds m1 in 74.0%, just under
the 75% the criterion asks (74.3% under the other channel rule). At matched command pulses (variant above, not deciding) m2 exceeds m1
in 94.9% of pre-MPTP and 97.1% of post-MPTP channel-sessions, so the pre-MPTP shortfall comes from
sessions with small pulses, whose m2 is near zero.

### trial-correlations

> Trial by trial, PTN response latencies did not correlate with the peak velocity or the m2 EMG
> response (Spearman |rho| < 0.34, p > 0.05).

The Methods define a response's onset on the trial-averaged SDF only. The trials a single-trial
latency would be needed for:

In [ ]:
ptn_resp = resp[resp.ptype == PTN]
n_trials = sum(int(studied.loc[(studied.file == r.file) & (studied.unit == r.unit), f"n_{r.direction}"].iloc[0])
               for r in ptn_resp.itertuples())
print(f"PTN responses (claims file's detection): {len(ptn_resp)}, over {n_trials} perturbation trials")

**Result: blocked (methods underspecified).** The criterion needs a single-trial response latency,
which the Methods do not define; it also uses the response selection blocked above.

## Summary

| Claim | Outcome |
|---|---|
| cell-counts | reproduced |
| monkey-l-perturbations | blocked (data missing) |
| baseline-rate | reproduced |
| ptn-faster-than-csn | reproduced |
| antidromic-latency | partially reproduced |
| false-positive-rate | blocked (methods underspecified) |
| 15 claims on the response selection | blocked (methods underspecified), results shown |
| kinematics-slower | not reproduced |
| kinematic-latency-unchanged | not reproduced |
| emg-m2-larger | reproduced |
| emg-duration-longer | partially reproduced |
| emg-onset-unchanged | not reproduced |
| emg-m1-weak | partially reproduced |
| trial-correlations | blocked (methods underspecified) |

A failure to reproduce here may be a failure of this attempt, not of the paper: see the
reasoning and the differences from the paper's method.

How the analysis differs from the paper's: monkey V only; units are this dataset's files' units;
the response detection is the claims file's, whose false-positive rate is about 20 times the
paper's, so no claim that rests on the response selection is decided; kinematic and EMG
parameters the paper does not give are the claims file's or standard choices (`pasq.py`).

Revisions after verification: the onset of the evoked movement was also measured with a causal
filter and with a below-threshold requirement, and all give shorter latencies after MPTP; the EMG
claims were repeated at matched torque commands, with the same outcomes; the claims on the
response selection were shown under the `trial_sd` reading, which meets the paper's
false-positive rate but finds about a quarter of the paper's responsive cells.

What was found about the dataset: the counts of cells by state and projection type match the
paper's monkey V table; no recording is deposited twice; the Leu files hold no perturbation
times; the torque command's pulses are about three times larger after MPTP than before, and the
evoked displacements (about 3 to 4 degrees) are smaller than the paper's 10 degrees and faster,
not slower, after MPTP; the angle's scale is consistent with the 20 degree targets and the file's
own velocity series, so this is in the data. Twelve files join two lab source files without
repeating trials. The post-MPTP Leu torque command moves at task events, not at perturbation
times.